# Basic SCVI pipeline

In [ ]:
import scanpy as sc
import scvi
import os
import sys

In [7]:
import os

# If SLURM set only total tasks, Lightning complains. Fix by unsetting or mapping.
if "SLURM_NTASKS" in os.environ and "SLURM_NTASKS_PER_NODE" not in os.environ:
    os.environ["SLURM_NTASKS_PER_NODE"] = os.environ["SLURM_NTASKS"]
# Safest: remove SLURM_NTASKS so Lightning doesn't read the unsupported one
os.environ.pop("SLURM_NTASKS", None)

# (Optional) if your notebook is single-node, be explicit:
os.environ.setdefault("SLURM_JOB_NODELIST", "localhost")
os.environ.setdefault("SLURM_JOB_NUM_NODES", "1")

'1'

In [11]:


def run_pipeline(h5ad_path):
    adata = sc.read_h5ad(h5ad_path)
    print(f"Loaded {h5ad_path} with {adata.n_obs} cells and {adata.n_vars} genes")

    # Basic QC
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    adata = adata[adata.obs.n_genes_by_counts < 2500, :]
    adata = adata[adata.obs.pct_counts_mt < 5, :]
    sc.pp.filter_genes(adata, min_cells=3)

    # Normalize and log-transform (optional with scVI but can be useful for inspection)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Prepare for scVI
    scvi.model.SCVI.setup_anndata(adata)

    # Train scVI
    model = scvi.model.SCVI(adata)
    model.train()

    # Add latent representation to AnnData
    adata.obsm["X_scVI"] = model.get_latent_representation()
    return adata

In [2]:
# print current pwd 
print("Current working directory:", os.getcwd())

Current working directory: /data1/peerd/riffled/riffled/Olaf_project/CARIBOU/benchmarking/basic_pipelines


In [9]:
FILE_PATH = "../../caribou/src/caribou/datasets/thymus_scrna-seq_atlas_-_myeloid_p2_subset.h5ad"  # Replace with your actual file path

In [12]:
adata = run_pipeline(FILE_PATH)

Loaded ../../caribou/src/caribou/datasets/thymus_scrna-seq_atlas_-_myeloid_p2_subset.h5ad with 843 cells and 36406 genes


/home/riffled/cfm/lib/python3.12/site-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number
/home/riffled/cfm/lib/python3.12/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/home/riffled/cfm/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/riffled/.local/lib/python3.12/site-packages/ip ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/riffled/cfm/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is 

Training:   0%|          | 0/400 [00:00<?, ?it/s]

/home/riffled/cfm/lib/python3.12/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/home/riffled/cfm/lib/python3.12/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/home/riffled/cfm/lib/python3.12/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/home/riffled/cfm/lib/python3.12/site-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/home/riffled/cfm/lib/python3.12/site-packages/scvi/module/_vae.py:573: UserWarning: The

In [17]:
adata

AnnData object with n_obs × n_vars = 704 × 12423
    obs: 'mapped_reference_annotation', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'organism_ontology_term_id', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_derivation_process', 'sample_source', 'tissue_type', 'suspension_depletion_factors', 'suspension_depleted_cell_types', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'library_uuid', 'assay_ontology_term_id', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'sample', 'enrichment', 'age_group', 'study', 'park_2020_cell_type', 'panfetal_2022_cell_type', 'cell_type_level_3', 'cell_type

# Run Metrics on basic SCVI

In [19]:
# --- New metric class using scib-metrics ------------------------------------
from scib_metrics.benchmark import Benchmarker
from typing import Dict
import anndata
import numpy as np

import json
from abc import ABC, abstractmethod

class AutoMetric(ABC):
    """
    Abstract base class for a metric to be applied to an AnnData object.
    """ 
    @abstractmethod
    def metric(self, adata) -> dict:
        """
        Run the metric and return a dictionary of results.
        """
        pass
    
    def run(self, adata):
        """
        Handles execution + JSON serialization.
        """
        result = self.metric(adata)
        print(json.dumps(result))  # Always print result at the end

EMBED = "X_scVI"        # The embedding key in adata.obsm
BATCH_KEY = "sample"     # The batch key in adata.obs
LABEL_KEY = "cell_type" # The cell type key in adata.obs

class IntegrationMetric(AutoMetric):
    """
    Compute SCIB integration quality metrics on an AnnData object using scib_metrics.
    Returns a dictionary with three metrics:
        • batch_silhouette: How well batches mix (lower ≈ better)
        • celltype_silhouette: How well cell types separate (higher ≈ better)
        • isolated_label_f1: Label preservation in isolated clusters (higher ≈ better)
    """
    def metric(self, adata):
        bm = Benchmarker(
            adata,
            batch_key=BATCH_KEY,
            label_key=LABEL_KEY,
            embedding_obsm_keys=[EMBED],        # list of embeddings to evaluate
        )
        bm.prepare()     # computes neighbors
        bm.benchmark()   # runs selected metrics
        results = bm.get_results()

        return results.to_dict()
    
IntegrationMetric().run(adata)

/home/riffled/cfm/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]/home/riffled/cfm/lib/python3.12/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)
Embeddings: 100%|██████████| 1/1 [00:15<00:00, 15.07s/it]


{"Isolated labels": {"X_scVI": 0.5347195267677307, "Metric Type": "Bio conservation"}, "KMeans NMI": {"X_scVI": 0.6081973941731372, "Metric Type": "Bio conservation"}, "KMeans ARI": {"X_scVI": 0.7385771348048894, "Metric Type": "Bio conservation"}, "Silhouette label": {"X_scVI": 0.5666902661323547, "Metric Type": "Bio conservation"}, "cLISI": {"X_scVI": 0.8960287570953369, "Metric Type": "Bio conservation"}, "BRAS": {"X_scVI": 0.6930656433105469, "Metric Type": "Batch correction"}, "iLISI": {"X_scVI": 0.07733388245105743, "Metric Type": "Batch correction"}, "KBET": {"X_scVI": 0.722659408563244, "Metric Type": "Batch correction"}, "Graph connectivity": {"X_scVI": 0.8465648587264402, "Metric Type": "Batch correction"}, "PCR comparison": {"X_scVI": 0.2787358133231578, "Metric Type": "Batch correction"}, "Batch correction": {"X_scVI": 0.523671918294657, "Metric Type": "Aggregate score"}, "Bio conservation": {"X_scVI": 0.6688426017761231, "Metric Type": "Aggregate score"}, "Total": {"X_scVI